In [ ]:
"""
Week 1: IoT Telemetry Ingestion and Signal Processing

Author: Subhashree Behera

Objective:
Create rolling-window statistical features from industrial sensor data
for predictive maintenance analysis.

Steps Performed:

1. Load fused AI4I dataset.
2. Sort records chronologically by date.
3. Select machine sensor columns.
4. Apply rolling window (window = 5).
5. Compute:

   * Rolling Mean
   * Rolling Standard Deviation
   * Rolling Variance
6. Generate 15 new engineered features.
7. Save the processed dataset for downstream analysis.

Output:
Feature-engineered dataset containing rolling statistics for:

* Air Temperature
* Process Temperature
* Rotational Speed
* Torque
* Tool Wear

Purpose:
Capture short-term operational trends and variability that may
indicate machine degradation or impending failure.
"""


In [1]:
# Step 1: Import required libraries
# pandas -> data manipulation, numpy -> numerical operations
import pandas as pd
import numpy as np

In [2]:
# Step 2: Load the fused dataset (sensors + weather, merged by date)
df = pd.read_csv('../data/ai4i_fused.csv')

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (10000, 21)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'date', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa']


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [3]:
# Step 3: Strip any accidental leading/trailing spaces from column names
# This prevents bugs later when referencing columns by exact name
df.columns = df.columns.str.strip()

print("Columns cleaned. Sample:", list(df.columns[:5]))

Columns cleaned. Sample: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]']


In [4]:
# Step 4: Check for null values before doing any feature engineering
# Rolling window calculations break if there are NaNs in the middle of data
print("Total null values:", df.isnull().sum().sum())

Total null values: 0


In [5]:
# Step 5: Sort by date (and UDI as tiebreaker) to ensure correct time-order
# Rolling window ONLY makes sense if data is in chronological sequence (last 5 readings)
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

print("First 3 dates after sorting:", df['date'].head(3).tolist())

First 3 dates after sorting: ['2020-01-01', '2020-01-01', '2020-01-01']


In [6]:
# Step 6: Define which sensor columns we'll engineer rolling features for
# These are the 5 raw machine sensor readings from the AI4I dataset
sensors = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

print(f"Will engineer rolling features for {len(sensors)} sensors:")
for s in sensors:
    print(" -", s)

Will engineer rolling features for 5 sensors:
 - Air temperature [K]
 - Process temperature [K]
 - Rotational speed [rpm]
 - Torque [Nm]
 - Tool wear [min]


In [7]:
# Step 7: Set the rolling window size
# window=5 means: for each row, look at the current + previous 4 readings
# min_periods=1 means: even the first few rows get a value (using fewer
# points), instead of NaN — avoids losing data at the start
window = 5

print(f"Rolling window size set to: {window}")

Rolling window size set to: 5


In [8]:
# Step 8: Calculate rolling MEAN for each sensor
# This smooths out short-term noise and shows the recent average trend
for col in sensors:
    df[f'{col}_rolling_mean'] = df[col].rolling(window=window, min_periods=1).mean()

print("Shape after adding rolling means:", df.shape)
df[['Torque [Nm]', 'Torque [Nm]_rolling_mean']].head(7)

Shape after adding rolling means: (10000, 26)


,Torque [Nm],Torque [Nm]_rolling_mean
0,42.8,42.800000
1,46.3,44.550000
2,49.4,46.166667
3,39.5,44.500000
4,40.0,43.600000
5,41.9,43.420000
6,42.4,42.640000


In [9]:
# Step 9: Calculate rolling STD for each sensor
# This measures how much the readings fluctuate in the recent window higher
# std = more unstable/erratic sensor behavior
for col in sensors:
    df[f'{col}_rolling_std'] = df[col].rolling(window=window, min_periods=1).std().fillna(0)

print("Shape after adding rolling std:", df.shape)

Shape after adding rolling std: (10000, 31)


In [10]:
# Step 10: Calculate rolling VARIANCE for each sensor
# Variance = std squared. It's another way to measure spread,
# often used directly in ML models since it amplifies larger fluctuations
for col in sensors:
    df[f'{col}_rolling_var'] = df[col].rolling(window=window, min_periods=1).var().fillna(0)

print("Shape after adding rolling variance:", df.shape)

Shape after adding rolling variance: (10000, 36)


In [11]:
# Step 11: Verify the final feature set
# We should have 5 sensors x 3 stats (mean, std, var) = 15 new columns
new_cols = [c for c in df.columns if 'rolling' in c]

print("Total new rolling features created:", len(new_cols))
print("Original columns: 21  ->  Final columns:", df.shape[1])
print("\nNew feature columns:")
for c in new_cols:
    print(" -", c)

Total new rolling features created: 15
Original columns: 21  ->  Final columns: 36

New feature columns:
 - Air temperature [K]_rolling_mean
 - Process temperature [K]_rolling_mean
 - Rotational speed [rpm]_rolling_mean
 - Torque [Nm]_rolling_mean
 - Tool wear [min]_rolling_mean
 - Air temperature [K]_rolling_std
 - Process temperature [K]_rolling_std
 - Rotational speed [rpm]_rolling_std
 - Torque [Nm]_rolling_std
 - Tool wear [min]_rolling_std
 - Air temperature [K]_rolling_var
 - Process temperature [K]_rolling_var
 - Rotational speed [rpm]_rolling_var
 - Torque [Nm]_rolling_var
 - Tool wear [min]_rolling_var


In [54]:
df.to_csv('../data/ai4i_rolling_features.csv', index=False)